In [2]:
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


g = 9.81

def y_at_x(theta, v0, x_target, g=g):
    """
    Вычисляет высоту снаряда y при достижении горизонтальной координаты x_target
    для заданных угла выстрела theta и начальной скорости v0.
    """
    c = math.cos(theta)
    t = x_target / (v0 * c)                    
    return v0 * math.sin(theta) * t - 0.5 * g * t * t 


def F(theta, v0, target):
    """
    разность между вычисленной и целевой высотой.
    """
    xt, yt = target
    return y_at_x(theta, v0, xt) - yt


In [3]:
def dF_dtheta(theta, v0, target, g=g):
    """
    Вычисляет производную dF/dθ 
    Используется в методе Ньютона для нахождения корня F(theta)=0.
    """
    x, y = target
    s = math.sin(theta)
    c = math.cos(theta)
    if abs(c) < 1e-12:
        return 0.0

    t = x / (v0 * c)  
    dt_dtheta = x * math.sin(theta) / (v0 * c * c) 
    dy_dtheta = (
        v0 * math.cos(theta) * t            
        + v0 * math.sin(theta) * dt_dtheta  
        - g * t * dt_dtheta                 
    )
    return dy_dtheta


In [4]:
import time
import math

def bisection_method(f, a, b, args=(), tol=1e-8, maxiter=100):
    """
    Метод бисекции.
    Делит отрезок [a, b] пополам, пока не найдёт корень уравнения f(x)=0 с заданной точностью.
    """
    start = time.perf_counter()
    fa = f(a, *args)
    fb = f(b, *args)
    if fa == 0:
        return a, 0, 0.0
    if fb == 0:
        return b, 0, 0.0

    it = 0
    while it < maxiter and (b - a) / 2.0 > tol:
        c = (a + b) / 2.0
        fc = f(c, *args)
        if fc == 0:
            a = b = c
            break
        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc
        it += 1

    duration = time.perf_counter() - start
    return (a + b) / 2.0, it, duration


def secant_method(f, x0, x1, args=(), tol=1e-10, maxiter=100):
    """
    Метод секущих.
    Аппроксимирует производную через две последние точки и уточняет корень уравнения f(x)=0.
    """
    start = time.perf_counter()
    it = 0
    f0 = f(x0, *args)
    f1 = f(x1, *args)
    while it < maxiter and abs(x1 - x0) > tol:
        if f1 == f0:
            break
        x2 = x1 - f1 * (x1 - x0) / (f1 - f0)
        x0, f0 = x1, f1
        x1 = x2
        f1 = f(x1, *args)
        it += 1
    duration = time.perf_counter() - start
    return x1, it, duration


def newton_method(f, df, x0, args=(), tol=1e-10, maxiter=100):
    """
    Метод Ньютона (касательных).
    Использует производную df(x) для уточнения корня f(x)=0.
    """
    start = time.perf_counter()
    x = x0
    it = 0
    while it < maxiter:
        fx = f(x, *args)
        if abs(fx) < tol:
            break
        dfx = df(x, *args)
        if dfx == 0:
            break
        x_new = x - fx / dfx
        if math.isnan(x_new) or math.isinf(x_new):
            break
        if abs(x_new - x) < tol:
            x = x_new
            break
        x = x_new
        it += 1
    duration = time.perf_counter() - start
    return x, it, duration




In [5]:
import math


theta_min = 1e-6
theta_max = math.pi / 2 - 1e-6

tests = [
    (30.0, (60.0, 0.0)),      
    (20.0, (30.0, 0.0)),      
    (40.0, (100.0, 0.0)),   
    (25.0, (40.0, 0.0)),      
    (25.0, (60, 0.0)),     
]



for (v0, target) in tests:
    print(f"Тест: v0={v0:.1f} м/с, цель={target}")
    print("-" * 70)

    # --- Бисекция ---
    theta_bis, it_bis, t_bis = bisection_method(
        F, theta_min, theta_max, args=(v0, target), tol=1e-10, maxiter=200
    )
    err_bis = abs(F(theta_bis, v0, target))
    print(f"  Bisection -> θ={math.degrees(theta_bis):8.4f}°, "
          f"iters={it_bis:3d}, resid={err_bis:.3e}, time={t_bis:.6f}s")

    # --- Метод секущих ---
    theta_sec, it_sec, t_sec = secant_method(
        F, theta_min, theta_max, args=(v0, target), tol=1e-12, maxiter=200
    )
    err_sec = abs(F(theta_sec, v0, target))
    print(f"  Secant    -> θ={math.degrees(theta_sec):8.4f}°, "
          f"iters={it_sec:3d}, resid={err_sec:.3e}, time={t_sec:.6f}s")

    # --- Метод Ньютона ---
    x0 = (theta_min + theta_max) / 2.0
    theta_newt, it_newt, t_newt = newton_method(
        F, dF_dtheta, x0, args=(v0, target), tol=1e-12, maxiter=100
    )
    err_newt = abs(F(theta_newt, v0, target))
    print(f"  Newton    -> θ={math.degrees(theta_newt):8.4f}°, "
          f"iters={it_newt:3d}, resid={err_newt:.3e}, time={t_newt:.6f}s")

    print()




Тест: v0=30.0 м/с, цель=(60.0, 0.0)
----------------------------------------------------------------------
  Bisection -> θ= 20.4219°, iters= 33, resid=5.865e-10, time=0.000018s
  Secant    -> θ= 20.4219°, iters=  8, resid=0.000e+00, time=0.000005s
  Newton    -> θ= 20.4219°, iters=  4, resid=2.132e-14, time=0.000007s

Тест: v0=20.0 м/с, цель=(30.0, 0.0)
----------------------------------------------------------------------
  Bisection -> θ= 23.6853°, iters= 33, resid=6.946e-10, time=0.000012s
  Secant    -> θ= 23.6853°, iters=  8, resid=3.553e-15, time=0.000004s
  Newton    -> θ= 23.6853°, iters=  5, resid=0.000e+00, time=0.000005s

Тест: v0=40.0 м/с, цель=(100.0, 0.0)
----------------------------------------------------------------------
  Bisection -> θ= 18.9079°, iters= 33, resid=5.186e-09, time=0.000010s
  Secant    -> θ= 18.9079°, iters=  8, resid=7.105e-15, time=0.000003s
  Newton    -> θ= 18.9079°, iters=  4, resid=7.105e-15, time=0.000004s

Тест: v0=25.0 м/с, цель=(40.0, 0.0)


Видно, метод секущих быстрее всех по времени, по итерациям- метод Ньютона.